In [ ]:
# Import Libraries
import ee
# import export
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

ee.Authenticate()
ee.Initialize(project='qsair-463811')

In [ ]:
# Define NDVI and Time Range
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

start_date = ee.Date('2020-04-01')
end_date = ee.Date('2020-09-30')

collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_indices)


In [ ]:
# Extract NDVI Time Series
def add_indices_s2(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    cloud_mask = img.select('QA60').bitwiseAnd(1 << 10).eq(0)  # simple QA60 mask
    return img.addBands(ndvi).updateMask(cloud_mask)

# rebuilt collection with cloud‑mask & higher CLOUDY_PIXEL threshold (40 %)
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
    .map(add_indices_s2)
)

def extract_field_features(field_geom, year, crop_type):
    start = ee.Date(f'{year}-04-01')
    end   = ee.Date(f'{year}-09-30')

    imgs = collection.filterBounds(field_geom).filterDate(start, end)

    def per_image(img):
        ndvi_val = img.select('NDVI').reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=field_geom,
            scale=10,
            bestEffort=True,
            maxPixels=1e9
        ).get('NDVI')

        return ee.Feature(None, {
            'doy': ee.Date(img.get('system:time_start')).getRelative('day', 'year'),
            'NDVI': ndvi_val
        })

    feats = imgs.map(per_image).filter(ee.Filter.notNull(['NDVI']))
    feat_list = feats.getInfo()['features']    # <-- always a list of dicts

    if not feat_list:                          # No clear images this season
        return {
            'peak_ndvi': np.nan,
            'early_ndvi': np.nan,
            'mid_season_ndvi': np.nan,
            'late_ndvi': np.nan,
            'crop_type': crop_type,
            'year': year
        }

    df = pd.DataFrame([f['properties'] for f in feat_list]).astype({'doy': int, 'NDVI': float})
    df = df.sort_values('doy')

    return {
        'peak_ndvi':          df['NDVI'].max(),
        'early_ndvi':         df.loc[df['doy'] == 120, 'NDVI'].iloc[0] if 120 in df['doy'].values else np.nan,
        'mid_season_ndvi':    df.loc[df['doy'] == 150, 'NDVI'].iloc[0] if 150 in df['doy'].values else np.nan,
        'late_ndvi':          df.loc[df['doy'] == 210, 'NDVI'].iloc[0] if 210 in df['doy'].values else np.nan,
        'crop_type':          crop_type,
        'year':               year
    }

In [ ]:
# Prepare Dataset
field_data = [
    {
        # Maize field – Kiryandongo District (north‑western Uganda)
        'geometry': ee.Geometry.Polygon([
            [31.7590, 1.6700],
            [31.7750, 1.6700],
            [31.7750, 1.6850],
            [31.7590, 1.6850]
        ]),
        'year': 2020,
        'crop_type': 'maize',
        'yield': 4.2          # tons/ha – replace with your actual number
    },
    {
        # Rice paddy – Butaleja District (eastern Uganda, Doho irrigation area)
        'geometry': ee.Geometry.Polygon([
            [33.5230, -0.5930],
            [33.5370, -0.5930],
            [33.5370, -0.5790],
            [33.5230, -0.5790]
        ]),
        'year': 2020,
        'crop_type': 'rice',
        'yield': 3.8          # tons/ha – example figure
    },
    {
        # Coffee plantation – Bushenyi District (south‑western Uganda)
        'geometry': ee.Geometry.Polygon([
            [30.6500, -0.6430],
            [30.6640, -0.6430],
            [30.6640, -0.6290],
            [30.6500, -0.6290]
        ]),
        'year': 2020,
        'crop_type': 'coffee',
        'yield': 2.1          # tons/ha – example figure
    }
]


features_list = []

for field in field_data:
    f = extract_field_features(field['geometry'], field['year'], field['crop_type'])
    f['yield'] = field['yield']
    features_list.append(f)

df = pd.DataFrame(features_list)
df = df.dropna()  # remove rows with missing NDVI


In [ ]:
print("Number of samples:", len(df))
print(df.head())


In [ ]:
# Train the model
X = df[['peak_ndvi', 'early_ndvi', 'mid_season_ndvi', 'late_ndvi', 'year', 'crop_type']]
X = pd.get_dummies(X)
y = df['yield']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model Performance:")
print("R²:", r2_score(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))


In [ ]:
# Feature Importance Plot
importances = model.feature_importances_
feature_names = X.columns

plt.figure(figsize=(8, 5))
plt.barh(feature_names, importances)
plt.xlabel("Importance")
plt.title("Feature Importance for Yield Prediction")
plt.tight_layout()
plt.show()


In [ ]:
# Make Prediction for New Field
new_field = {
    'geometry': ee.Geometry.Polygon([...]),  # Replace with coordinates
    'year': 2020,
    'crop_type': 'maize'
}

new_features = extract_field_features(new_field['geometry'], new_field['year'], new_field['crop_type'])
new_X = pd.DataFrame([new_features])
new_X = pd.get_dummies(new_X).reindex(columns=X.columns, fill_value=0)

predicted_yield = model.predict(new_X)[0]
print(f"Predicted yield for new field: {predicted_yield:.2f} tons/ha")
